# 77. The Dynamic & Real-Time VRP
## Tier 6: The Autonomous & Self-Optimizing Ecosystem (Distributed Intelligence)

### Key assumptions
- Multiple autonomous agents with specialized capabilities and decision-making authority
- Contract Net Protocol for efficient request assignment and negotiation
- Local agent interactions create emergent global optimization
- Zone-based geographical organization for scalable coordination
- Real-time communication and negotiation protocols

### Approach (step-by-step)
1. **Design multi-agent architecture** with Vehicle, Zone, Request, and Infrastructure agents
2. **Implement Contract Net Protocol** for request assignment and bidding
3. **Create bid calculation methods** with multi-criteria evaluation
4. **Develop negotiation algorithms** for conflict resolution and coordination
5. **Implement emergent optimization** through local agent interactions
6. **Simulate distributed coordination** scenarios with real-time decision making

### What to look for in the results
- Emergent system behavior from local interactions
- Negotiation overhead and communication efficiency
- Dynamic load balancing across vehicles
- Anticipatory positioning through zone agent coordination
- System-wide performance improvements

### Concrete example (from the source)
Metropolitan delivery system with 15 vehicles across 4 zones:
- 23 active requests during peak lunch-hour demand
- 8 new arrivals in 5 minutes at 12:30 PM
- Urgent pharmaceutical delivery request processing
- 47ms total negotiation overhead
- 15.3% reduction in average delivery time, 8.7% improvement in vehicle utilization

### Visualization(s)
- Multi-agent communication and negotiation flows
- Zone-based vehicle distribution and request assignments
- Emergent optimization patterns over time
- Performance comparison with centralized approaches

### Why this Tier exists vs earlier Tiers
Tier 6 addresses limitations of previous approaches by:
- **Scalability**: Distributed coordination handles large-scale systems
- **Resilience**: No single point of failure through distributed intelligence
- **Adaptability**: Local agents respond quickly to changing conditions
- **Emergence**: Global optimization emerges from local decisions

### Pros / Cons vs Tier 1 & Tier 4
**Pros:**
- Scales to hundreds of vehicles and thousands of requests
- Real-time adaptation without central coordination bottleneck
- Resilient to individual agent failures
- Natural load balancing and resource optimization

**Cons:**
- No global optimality guarantees
- Complex coordination and communication protocols
- Potential for conflicting local decisions
- Requires sophisticated agent design and negotiation

### When to use this Tier
- Large-scale metropolitan delivery systems
- Dynamic environments requiring rapid adaptation
- Systems with geographical distribution
- Applications requiring high resilience and scalability
- When centralized coordination becomes a bottleneck

In [1]:
# Import required libraries for distributed intelligence approach
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional, Set
from enum import Enum
import random
import time
from collections import defaultdict, deque
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

In [ ]:
class AgentType(Enum):
    """Types of autonomous agents in the distributed system"""
    VEHICLE = "vehicle"
    ZONE = "zone"
    REQUEST = "request"
    INFRASTRUCTURE = "infrastructure"

@dataclass
class Message:
    """Communication message between agents"""
    sender_id: str
    receiver_id: str
    message_type: str
    content: Dict
    timestamp: float
    priority: int = 1  # 1=low, 2=medium, 3=high

@dataclass
class Bid:
    """Bid submitted by vehicle agent for request assignment"""
    vehicle_id: str
    request_id: str
    bid_value: float
    detour_cost: float
    time_compliance: float
    capacity_utilization: float
    future_impact: float
    confidence: float

@dataclass
class ServiceRequest:
    """Service request represented as autonomous agent"""
    id: str
    origin: Tuple[int, int]
    destination: Tuple[int, int]
    demand: int
    time_window_early: float
    time_window_late: float
    priority: int  # 1=high, 2=medium, 3=low
    arrival_time: float
    assigned_vehicle: Optional[str] = None
    status: str = "pending"  # pending, assigned, in_progress, completed

print("Agent communication data structures defined")

In [ ]:
class VehicleAgent:
    """Autonomous vehicle agent with local routing intelligence"""
    
    def __init__(self, vehicle_id: str, initial_position: Tuple[int, int], 
                 capacity: int, zone_id: str):
        self.id = vehicle_id
        self.position = initial_position
        self.capacity = capacity
        self.remaining_capacity = capacity
        self.zone_id = zone_id
        self.current_route: List[Tuple[int, int]] = []
        self.assigned_requests: List[ServiceRequest] = []
        self.traffic_awareness: Dict = {}
        self.local_performance_metrics: Dict = {
            'total_distance': 0.0,
            'completed_requests': 0,
            'average_response_time': 0.0
        }
    
    def calculate_bid(self, request: ServiceRequest, 
                     traffic_conditions: Dict) -> Bid:
        """Calculate bid for serving a request using multi-criteria evaluation"""
        
        # Calculate detour cost
        detour_cost = self._calculate_detour_cost(request)
        
        # Calculate time window compliance
        time_to_origin = self._estimate_travel_time(self.position, request.origin, traffic_conditions)
        arrival_time = time.time() + time_to_origin
        time_compliance = 1.0 if request.time_window_early <= arrival_time <= request.time_window_late else 0.5
        
        # Calculate capacity utilization
        capacity_utilization = (self.capacity - self.remaining_capacity + request.demand) / self.capacity
        
        # Estimate future impact on service capability
        future_impact = self._estimate_future_impact(request)
        
        # Calculate bid value (lower is better for request agent)
        w1, w2, w3, w4 = 0.3, 0.25, 0.25, 0.2  # Weight coefficients
        bid_value = (w1 * detour_cost + 
                    w2 * (2.0 - time_compliance) +  # Convert to cost (higher compliance = lower cost)
                    w3 * capacity_utilization +
                    w4 * future_impact)
        
        # Calculate confidence in bid
        confidence = min(1.0, (time_compliance + (1.0 - capacity_utilization)) / 2.0)
        
        return Bid(
            vehicle_id=self.id,
            request_id=request.id,
            bid_value=bid_value,
            detour_cost=detour_cost,
            time_compliance=time_compliance,
            capacity_utilization=capacity_utilization,
            future_impact=future_impact,
            confidence=confidence
        )
    
    def _calculate_detour_cost(self, request: ServiceRequest) -> float:
        """Calculate detour cost for adding request to current route"""
        # Simplified detour calculation
        current_route_cost = self._calculate_route_cost()
        
        # New route with request inserted (simplified: insert at optimal position)
        new_route = self.current_route + [request.origin, request.destination]
        new_route_cost = self._calculate_route_cost_for_route(new_route)
        
        return new_route_cost - current_route_cost
    
    def _calculate_route_cost(self) -> float:
        """Calculate cost of current route"""
        return self._calculate_route_cost_for_route(self.current_route)
    
    def _calculate_route_cost_for_route(self, route: List[Tuple[int, int]]) -> float:
        """Calculate total cost for a given route"""
        if len(route) < 2:
            return 0.0
        
        total_cost = 0.0
        current_pos = self.position
        
        for next_pos in route:
            total_cost += np.sqrt((current_pos[0] - next_pos[0])**2 + 
                                 (current_pos[1] - next_pos[1])**2)
            current_pos = next_pos
        
        return total_cost
    
    def _estimate_travel_time(self, origin: Tuple[int, int], 
                             destination: Tuple[int, int], 
                             traffic_conditions: Dict) -> float:
        """Estimate travel time considering traffic conditions"""
        distance = np.sqrt((origin[0] - destination[0])**2 + 
                          (origin[1] - destination[1])**2)
        
        # Apply traffic factor
        congestion_level = traffic_conditions.get('congestion', 0.5)
        traffic_factor = 1.0 + congestion_level * 0.5  # Up to 50% slower
        
        return distance * traffic_factor
    
    def _estimate_future_impact(self, request: ServiceRequest) -> float:
        """Estimate impact of accepting request on future service capability"""
        # Simplified: based on remaining capacity and current workload
        workload_factor = len(self.assigned_requests) / 10.0  # Normalize to 0-1
        capacity_factor = (self.capacity - self.remaining_capacity + request.demand) / self.capacity
        
        return (workload_factor + capacity_factor) / 2.0
    
    def assign_request(self, request: ServiceRequest):
        """Assign a request to this vehicle"""
        self.assigned_requests.append(request)
        self.remaining_capacity -= request.demand
        request.assigned_vehicle = self.id
        request.status = "assigned"
        
        # Update route (simplified)
        self.current_route.extend([request.origin, request.destination])

print("Vehicle agent implementation completed")

In [ ]:
class ZoneAgent:
    """Zone agent managing geographical regions and vehicle coordination"""
    
    def __init__(self, zone_id: str, boundaries: Tuple[int, int, int, int]):
        self.id = zone_id
        self.boundaries = boundaries  # (x_min, y_min, x_max, y_max)
        self.vehicles: List[VehicleAgent] = []
        self.demand_forecast: Dict = {}
        self.request_patterns: List = []
        self.coordination_history: List = []
        
    def add_vehicle(self, vehicle: VehicleAgent):
        """Add vehicle to this zone"""
        self.vehicles.append(vehicle)
        vehicle.zone_id = self.id
    
    def update_demand_forecast(self, recent_requests: List[ServiceRequest]):
        """Update demand forecast based on recent request patterns"""
        # Count requests in this zone
        zone_requests = [r for r in recent_requests if self._is_in_zone(r.origin)]
        
        # Simple demand forecasting (moving average)
        current_demand = len(zone_requests)
        historical_avg = self.demand_forecast.get('average', current_demand)
        
        # Update forecast with exponential smoothing
        alpha = 0.3  # Smoothing factor
        new_average = alpha * current_demand + (1 - alpha) * historical_avg
        
        self.demand_forecast = {
            'average': new_average,
            'current': current_demand,
            'trend': current_demand - historical_avg
        }
    
    def guide_idle_vehicles(self):
        """Guide idle vehicles toward high-demand areas"""
        idle_vehicles = [v for v in self.vehicles if len(v.assigned_requests) < 2]
        
        if not idle_vehicles or self.demand_forecast.get('trend', 0) <= 0:
            return
        
        # Calculate high-demand center (simplified)
        demand_center = self._calculate_demand_center()
        
        for vehicle in idle_vehicles:
            # Suggest repositioning toward demand center
            distance_to_center = np.sqrt((vehicle.position[0] - demand_center[0])**2 + 
                                         (vehicle.position[1] - demand_center[1])**2)
            
            if distance_to_center > 20:  # Only if far from center
                # Create repositioning suggestion (simplified)
                self._suggest_repositioning(vehicle, demand_center)
    
    def coordinate_cross_zone_requests(self, request: ServiceRequest, 
                                      other_zones: List['ZoneAgent']) -> Optional[VehicleAgent]:
        """Coordinate requests that span multiple zones"""
        # Find best vehicle across all zones
        all_vehicles = []
        for zone in [self] + other_zones:
            all_vehicles.extend(zone.vehicles)
        
        best_vehicle = None
        best_bid_value = float('inf')
        
        for vehicle in all_vehicles:
            # Check if vehicle can serve request
            if vehicle.remaining_capacity >= request.demand:
                bid = vehicle.calculate_bid(request, {'congestion': 0.5})
                
                if bid.bid_value < best_bid_value:
                    best_bid_value = bid.bid_value
                    best_vehicle = vehicle
        
        return best_vehicle
    
    def _is_in_zone(self, position: Tuple[int, int]) -> bool:
        """Check if position is within zone boundaries"""
        x_min, y_min, x_max, y_max = self.boundaries
        return x_min <= position[0] <= x_max and y_min <= position[1] <= y_max
    
    def _calculate_demand_center(self) -> Tuple[int, int]:
        """Calculate center of high demand in this zone"""
        # Simplified: return zone center
        x_min, y_min, x_max, y_max = self.boundaries
        return ((x_min + x_max) // 2, (y_min + y_max) // 2)
    
    def _suggest_repositioning(self, vehicle: VehicleAgent, target: Tuple[int, int]):
        """Suggest vehicle repositioning (simplified implementation)"""
        # In a real implementation, this would send a message to the vehicle
        pass

print("Zone agent implementation completed")

In [ ]:
class RequestAgent:
    """Autonomous request agent representing service demands"""
    
    def __init__(self, request: ServiceRequest):
        self.request = request
        self.id = request.id
        self.bid_history: List[Bid] = []
        self.negotiation_rounds = 0
        self.max_negotiation_rounds = 3
        self.assignment_status = "searching"
    
    def initiate_bidding_process(self, vehicle_agents: List[VehicleAgent], 
                                traffic_conditions: Dict) -> List[Bid]:
        """Initiate bidding process for service assignment"""
        bids = []
        
        # Broadcast to vehicles within feasible range
        feasible_vehicles = self._find_feasible_vehicles(vehicle_agents)
        
        for vehicle in feasible_vehicles:
            bid = vehicle.calculate_bid(self.request, traffic_conditions)
            bids.append(bid)
            self.bid_history.append(bid)
        
        return bids
    
    def evaluate_bids(self, bids: List[Bid]) -> Optional[VehicleAgent]:
        """Evaluate received bids and select best vehicle"""
        if not bids:
            return None
        
        # Sort by bid value (lower is better)
        sorted_bids = sorted(bids, key=lambda b: b.bid_value)
        
        # Select best bid with sufficient confidence
        for bid in sorted_bids:
            if bid.confidence >= 0.7:  # Minimum confidence threshold
                return bid.vehicle_id
        
        # If no bid meets confidence threshold, select best anyway
        return sorted_bids[0].vehicle_id
    
    def negotiate_assignment(self, selected_vehicle_id: str, 
                           vehicle_agents: List[VehicleAgent]) -> bool:
        """Negotiate final assignment with selected vehicle"""
        # Find the selected vehicle
        selected_vehicle = None
        for vehicle in vehicle_agents:
            if vehicle.id == selected_vehicle_id:
                selected_vehicle = vehicle
                break
        
        if not selected_vehicle:
            return False
        
        # Check if vehicle still has capacity
        if selected_vehicle.remaining_capacity < self.request.demand:
            return False
        
        # Finalize assignment
        selected_vehicle.assign_request(self.request)
        self.assignment_status = "assigned"
        
        return True
    
    def _find_feasible_vehicles(self, vehicle_agents: List[VehicleAgent]) -> List[VehicleAgent]:
        """Find vehicles that can potentially serve this request"""
        feasible_vehicles = []
        
        for vehicle in vehicle_agents:
            # Check capacity constraint
            if vehicle.remaining_capacity >= self.request.demand:
                # Check time window feasibility (simplified)
                distance = np.sqrt((vehicle.position[0] - self.request.origin[0])**2 + 
                                 (vehicle.position[1] - self.request.origin[1])**2)
                estimated_time = distance * 2  # Simplified time estimation
                
                current_time = time.time()
                arrival_time = current_time + estimated_time
                
                if self.request.time_window_early <= arrival_time <= self.request.time_window_late:
                    feasible_vehicles.append(vehicle)
        
        return feasible_vehicles

print("Request agent implementation completed")

In [ ]:
class DistributedDVRPSystem:
    """Multi-agent distributed DVRP coordination system"""
    
    def __init__(self, num_zones: int = 4):
        self.zone_agents: List[ZoneAgent] = []
        self.vehicle_agents: List[VehicleAgent] = []
        self.request_agents: List[RequestAgent] = []
        self.message_queue: deque = deque()
        self.system_metrics: Dict = {
            'total_requests_processed': 0,
            'average_response_time': 0.0,
            'vehicle_utilization': 0.0,
            'negotiation_overhead': 0.0
        }
        
        # Initialize zones
        self._initialize_zones(num_zones)
    
    def _initialize_zones(self, num_zones: int):
        """Initialize geographical zones"""
        # Create zones in a 2x2 grid for simplicity
        zone_size = 50
        
        for i in range(num_zones):
            row = i // 2
            col = i % 2
            
            x_min = col * zone_size
            y_min = row * zone_size
            x_max = x_min + zone_size
            y_max = y_min + zone_size
            
            zone = ZoneAgent(f"zone_{i}", (x_min, y_min, x_max, y_max))
            self.zone_agents.append(zone)
    
    def add_vehicles(self, vehicles_config: List[Dict]):
        """Add vehicles to the system"""
        for config in vehicles_config:
            vehicle = VehicleAgent(
                vehicle_id=config['id'],
                initial_position=config['position'],
                capacity=config['capacity'],
                zone_id=config.get('zone_id', 'zone_0')
            )
            
            self.vehicle_agents.append(vehicle)
            
            # Add to appropriate zone
            for zone in self.zone_agents:
                if zone._is_in_zone(vehicle.position):
                    zone.add_vehicle(vehicle)
                    break
    
    def process_request(self, request: ServiceRequest) -> Dict:
        """Process a new service request through distributed coordination"""
        start_time = time.time()
        
        # Create request agent
        request_agent = RequestAgent(request)
        self.request_agents.append(request_agent)
        
        # Get current traffic conditions
        traffic_conditions = {'congestion': random.uniform(0.1, 0.9)}
        
        # Initiate bidding process
        bids = request_agent.initiate_bidding_process(self.vehicle_agents, traffic_conditions)
        
        # Evaluate bids and select vehicle
        selected_vehicle_id = request_agent.evaluate_bids(bids)
        
        # Negotiate assignment
        assignment_success = False
        if selected_vehicle_id:
            assignment_success = request_agent.negotiate_assignment(selected_vehicle_id, self.vehicle_agents)
        
        # Calculate processing time
        processing_time = time.time() - start_time
        
        # Update metrics
        self.system_metrics['total_requests_processed'] += 1
        self.system_metrics['negotiation_overhead'] += processing_time
        
        return {
            'request_id': request.id,
            'assigned_vehicle': selected_vehicle_id,
            'assignment_success': assignment_success,
            'processing_time': processing_time,
            'bids_received': len(bids),
            'negotiation_rounds': request_agent.negotiation_rounds
        }
    
    def optimize_system_performance(self):
        """Run system-wide optimization through local agent interactions"""
        # Update demand forecasts in all zones
        recent_requests = [ra.request for ra in self.request_agents[-10:]]  # Last 10 requests
        
        for zone in self.zone_agents:
            zone.update_demand_forecast(recent_requests)
            zone.guide_idle_vehicles()
        
        # Calculate system metrics
        self._update_system_metrics()
    
    def _update_system_metrics(self):
        """Update system performance metrics"""
        if self.system_metrics['total_requests_processed'] > 0:
            self.system_metrics['average_response_time'] = (
                self.system_metrics['negotiation_overhead'] / 
                self.system_metrics['total_requests_processed']
            ) * 1000  # Convert to milliseconds
        
        # Calculate vehicle utilization
        total_capacity = sum(v.capacity for v in self.vehicle_agents)
        used_capacity = sum(v.capacity - v.remaining_capacity for v in self.vehicle_agents)
        
        if total_capacity > 0:
            self.system_metrics['vehicle_utilization'] = used_capacity / total_capacity

print("Distributed DVRP system implementation completed")

In [ ]:
def create_metropolitan_scenario() -> DistributedDVRPSystem:
    """Create the metropolitan delivery scenario from the source material"""
    
    print("🏙️  Creating Metropolitan Delivery Scenario")
    print("=" * 50)
    
    # Initialize distributed system with 4 zones
    system = DistributedDVRPSystem(num_zones=4)
    
    # Add 15 vehicles across 4 zones (3-4 vehicles per zone)
    vehicles_config = []
    vehicle_id = 0
    
    for zone_idx in range(4):
        num_vehicles = 4 if zone_idx < 3 else 3  # First 3 zones have 4 vehicles, last has 3
        
        for i in range(num_vehicles):
            # Random position within zone
            x_min = (zone_idx % 2) * 50
            y_min = (zone_idx // 2) * 50
            position = (
                random.randint(x_min, x_min + 49),
                random.randint(y_min, y_min + 49)
            )
            
            vehicles_config.append({
                'id': f'vehicle_{vehicle_id}',
                'position': position,
                'capacity': random.randint(15, 25),
                'zone_id': f'zone_{zone_idx}'
            })
            
            vehicle_id += 1
    
    system.add_vehicles(vehicles_config)
    
    print(f"✅ Created system with {len(system.vehicle_agents)} vehicles across {len(system.zone_agents)} zones")
    
    # Display zone information
    for zone in system.zone_agents:
        print(f"  📍 {zone.id}: {len(zone.vehicles)} vehicles, boundaries {zone.boundaries}")
    
    return system

# Create the scenario
system = create_metropolitan_scenario()

In [ ]:
def simulate_peak_hour_scenario(system: DistributedDVRPSystem) -> Dict:
    """Simulate the peak hour scenario with 23 active requests"""
    
    print("\n🕐 Simulating Peak Hour Scenario (12:30 PM)")
    print("=" * 45)
    
    # Generate 23 active requests with 8 new arrivals
    requests = []
    current_time = 12.5 * 60  # 12:30 PM in minutes
    
    # Create existing requests (15)
    for i in range(15):
        request = ServiceRequest(
            id=f"req_existing_{i}",
            origin=(random.randint(0, 99), random.randint(0, 99)),
            destination=(random.randint(0, 99), random.randint(0, 99)),
            demand=random.randint(1, 4),
            time_window_early=current_time + random.uniform(-30, 30),
            time_window_late=current_time + random.uniform(30, 120),
            priority=random.choices([1, 2, 3], weights=[0.2, 0.5, 0.3])[0],
            arrival_time=current_time - random.uniform(5, 60)
        )
        requests.append(request)
    
    # Create new arrivals (8)
    for i in range(8):
        request = ServiceRequest(
            id=f"req_new_{i}",
            origin=(random.randint(0, 99), random.randint(0, 99)),
            destination=(random.randint(0, 99), random.randint(0, 99)),
            demand=random.randint(1, 4),
            time_window_early=current_time + random.uniform(15, 45),
            time_window_late=current_time + random.uniform(45, 120),
            priority=random.choices([1, 2, 3], weights=[0.3, 0.4, 0.3])[0],
            arrival_time=current_time
        )
        requests.append(request)
    
    print(f"📦 Generated {len(requests)} total requests ({len([r for r in requests if r.arrival_time == current_time])} new arrivals)")
    
    # Process requests through distributed coordination
    processing_results = []
    
    for i, request in enumerate(requests):
        print(f"\n🔄 Processing request {i+1}/{len(requests)}: {request.id}")
        
        result = system.process_request(request)
        processing_results.append(result)
        
        status = "✅ ASSIGNED" if result['assignment_success'] else "❌ FAILED"
        print(f"   {status} to {result['assigned_vehicle']} in {result['processing_time']*1000:.1f}ms")
        print(f"   📊 Received {result['bids_received']} bids")
    
    # Run system optimization
    print("\n🔧 Running system-wide optimization...")
    system.optimize_system_performance()
    
    return {
        'requests_processed': len(requests),
        'successful_assignments': sum(1 for r in processing_results if r['assignment_success']),
        'processing_results': processing_results,
        'system_metrics': system.system_metrics.copy()
    }

# Run the peak hour simulation
simulation_results = simulate_peak_hour_scenario(system)

In [ ]:
def simulate_pharmaceutical_urgency_scenario(system: DistributedDVRPSystem) -> Dict:
    """Simulate the urgent pharmaceutical delivery scenario"""
    
    print("\n🚨 Simulating Urgent Pharmaceutical Delivery Scenario")
    print("=" * 55)
    
    # Create urgent pharmaceutical request
    urgent_request = ServiceRequest(
        id="pharmacy_urgent_001",
        origin=(25, 35),  # In Zone C
        destination=(65, 75),  # In Zone D
        demand=2,
        time_window_early=12.5 * 60,  # 12:30 PM
        time_window_late=13.0 * 60,   # 1:00 PM (30-minute window)
        priority=1,  # High priority
        arrival_time=12.5 * 60
    )
    
    print(f"📋 Urgent request: {urgent_request.origin} → {urgent_request.destination}")
    print(f"⏰ Time window: [{urgent_request.time_window_early/60:.1f}, {urgent_request.time_window_late/60:.1f}]")
    print(f"🎯 Priority: HIGH (Pharmaceutical delivery)")
    
    # Track negotiation steps
    negotiation_log = []
    
    # Step 1: Request agent initiates bidding
    print("\n📢 Step 1: Request Agent Initiates Bidding Process")
    request_agent = RequestAgent(urgent_request)
    
    traffic_conditions = {'congestion': 0.6}  # Moderate traffic
    bids = request_agent.initiate_bidding_process(system.vehicle_agents, traffic_conditions)
    
    print(f"   📊 Received {len(bids)} bids from vehicles")
    negotiation_log.append(f"Bidding initiated: {len(bids)} bids received")
    
    # Step 2: Evaluate bids and select best
    print("\n🤔 Step 2: Evaluating Bids and Selecting Best Vehicle")
    
    # Show top 3 bids
    sorted_bids = sorted(bids, key=lambda b: b.bid_value)[:3]
    
    for i, bid in enumerate(sorted_bids, 1):
        print(f"   {i}. Vehicle {bid.vehicle_id}: Bid={bid.bid_value:.2f}, Confidence={bid.confidence:.2f}")
        print(f"      Detour={bid.detour_cost:.2f}, Time Compliance={bid.time_compliance:.2f}")
    
    selected_vehicle_id = request_agent.evaluate_bids(bids)
    negotiation_log.append(f"Vehicle selected: {selected_vehicle_id}")
    
    print(f"\n✅ Selected Vehicle: {selected_vehicle_id}")
    
    # Step 3: Negotiate final assignment
    print("\n🤝 Step 3: Final Assignment Negotiation")
    
    assignment_success = request_agent.negotiate_assignment(selected_vehicle_id, system.vehicle_agents)
    
    if assignment_success:
        print("   ✅ Assignment successful!")
        negotiation_log.append("Assignment successful")
        
        # Step 4: Secondary negotiation for load balancing
        print("\n⚖️  Step 4: Load Balancing Negotiation")
        
        # Find the assigned vehicle
        assigned_vehicle = None
        for vehicle in system.vehicle_agents:
            if vehicle.id == selected_vehicle_id:
                assigned_vehicle = vehicle
                break
        
        if assigned_vehicle and len(assigned_vehicle.assigned_requests) > 1:
            print(f"   🔄 Vehicle {selected_vehicle_id} has {len(assigned_vehicle.assigned_requests)} requests")
            print(f"   💡 Initiating secondary negotiation for load balancing...")
            negotiation_log.append("Load balancing negotiation initiated")
        else:
            print("   ⚖️  No load balancing needed")
            negotiation_log.append("No load balancing needed")
    
    else:
        print("   ❌ Assignment failed - insufficient capacity")
        negotiation_log.append("Assignment failed")
    
    return {
        'urgent_request': urgent_request,
        'selected_vehicle': selected_vehicle_id,
        'assignment_success': assignment_success,
        'bids_received': len(bids),
        'negotiation_log': negotiation_log,
        'best_bids': sorted_bids
    }

# Run the pharmaceutical urgency scenario
pharmacy_results = simulate_pharmaceutical_urgency_scenario(system)

In [ ]:
def visualize_distributed_system_results(simulation_results: Dict, 
                                        pharmacy_results: Dict,
                                        system: DistributedDVRPSystem):
    """Create comprehensive visualization of distributed system results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Distributed Intelligence DVRP - Multi-Agent Coordination Results', 
                 fontsize=16, fontweight='bold')
    
    # Plot 1: Zone-based vehicle distribution
    ax1 = axes[0, 0]
    ax1.set_title('Zone-Based Vehicle Distribution', fontweight='bold')
    ax1.set_xlabel('Zone')
    ax1.set_ylabel('Number of Vehicles')
    
    zone_names = [f"Zone {i.split('_')[1]}" for i in range(len(system.zone_agents))]
    vehicle_counts = [len(zone.vehicles) for zone in system.zone_agents]
    
    bars = ax1.bar(zone_names, vehicle_counts, color='lightblue', alpha=0.7, edgecolor='black')
    
    # Add value labels on bars
    for bar, count in zip(bars, vehicle_counts):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                str(count), ha='center', va='bottom', fontweight='bold')
    
    ax1.grid(True, alpha=0.3, axis='y')
    
    # Plot 2: Request processing performance
    ax2 = axes[0, 1]
    ax2.set_title('Request Processing Performance', fontweight='bold')
    ax2.set_xlabel('Request ID')
    ax2.set_ylabel('Processing Time (ms)')
    
    processing_times = [r['processing_time'] * 1000 for r in simulation_results['processing_results']]
    request_ids = list(range(1, len(processing_times) + 1))
    
    # Color code by success/failure
    colors = ['green' if r['assignment_success'] else 'red' 
             for r in simulation_results['processing_results']]
    
    ax2.scatter(request_ids, processing_times, c=colors, alpha=0.7, s=50)
    ax2.axhline(y=np.mean(processing_times), color='blue', linestyle='--', 
               label=f'Average: {np.mean(processing_times):.1f}ms')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: System performance metrics
    ax3 = axes[1, 0]
    ax3.set_title('System Performance Metrics', fontweight='bold')
    ax3.set_ylabel('Performance Value')
    
    metrics = {
        'Success Rate': simulation_results['successful_assignments'] / simulation_results['requests_processed'],
        'Vehicle Utilization': system.system_metrics['vehicle_utilization'],
        'Avg Response Time (s)': system.system_metrics['average_response_time'] / 1000
    }
    
    metric_names = list(metrics.keys())
    metric_values = list(metrics.values())
    
    bars = ax3.bar(metric_names, metric_values, color=['lightgreen', 'orange', 'lightcoral'], 
                    alpha=0.7, edgecolor='black')
    
    # Add value labels
    for bar, value in zip(bars, metric_values):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{value:.3f}', ha='center', va='bottom', fontweight='bold')
    
    ax3.grid(True, alpha=0.3, axis='y')
    plt.setp(ax3.get_xticklabels(), rotation=45, ha='right')
    
    # Plot 4: Urgent pharmaceutical scenario details
    ax4 = axes[1, 1]
    ax4.set_title('Urgent Pharmaceutical Scenario', fontweight='bold')
    ax4.axis('off')
    
    # Create scenario summary text
    scenario_text = f"""
🚨 URGENT PHARMACEUTICAL DELIVERY:
  • Request: {pharmacy_results['urgent_request'].origin} → {pharmacy_results['urgent_request'].destination}
  • Time Window: [{pharmacy_results['urgent_request'].time_window_early/60:.1f}, {pharmacy_results['urgent_request'].time_window_late/60:.1f}]
  • Priority: HIGH
  • Bids Received: {pharmacy_results['bids_received']}
  • Selected Vehicle: {pharmacy_results['selected_vehicle']}
  • Assignment: {'✅ SUCCESS' if pharmacy_results['assignment_success'] else '❌ FAILED'}

📊 TOP 3 BIDS:
"""
    
    for i, bid in enumerate(pharmacy_results['best_bids'][:3], 1):
        scenario_text += f"\n  {i}. Vehicle {bid.vehicle_id}: {bid.bid_value:.2f} (confidence: {bid.confidence:.2f})"
    
    scenario_text += f"""

🔄 NEGOTIATION STEPS:
"""
    
    for i, log_entry in enumerate(pharmacy_results['negotiation_log'], 1):
        scenario_text += f"\n  {i}. {log_entry}"
    
    ax4.text(0.05, 0.95, scenario_text, transform=ax4.transAxes, fontsize=9,
            verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# Visualize the results
visualize_distributed_system_results(simulation_results, pharmacy_results, system)

In [ ]:
def analyze_emergent_optimization(system: DistributedDVRPSystem) -> Dict:
    """Analyze emergent optimization patterns from local interactions"""
    
    print("\n🔍 ANALYSIS OF EMERGENT OPTIMIZATION PATTERNS")
    print("=" * 55)
    
    analysis = {
        'load_balancing': {},
        'anticipatory_positioning': {},
        'adaptive_coordination': {},
        'system_resilience': {}
    }
    
    # Analyze load balancing
    vehicle_workloads = [len(v.assigned_requests) for v in system.vehicle_agents]
    workload_std = np.std(vehicle_workloads)
    workload_mean = np.mean(vehicle_workloads)
    
    analysis['load_balancing'] = {
        'mean_workload': workload_mean,
        'workload_std': workload_std,
        'balance_coefficient': workload_std / max(workload_mean, 1),
        'idle_vehicles': sum(1 for w in vehicle_workloads if w == 0),
        'overloaded_vehicles': sum(1 for w in vehicle_workloads if w > workload_mean + workload_std)
    }
    
    print(f"📊 LOAD BALANCING:")
    print(f"  • Average workload: {workload_mean:.2f} requests/vehicle")
    print(f"  • Workload variance: {workload_std:.2f}")
    print(f"  • Balance coefficient: {analysis['load_balancing']['balance_coefficient']:.3f}")
    print(f"  • Idle vehicles: {analysis['load_balancing']['idle_vehicles']}")
    print(f"  • Overloaded vehicles: {analysis['load_balancing']['overloaded_vehicles']}")
    
    # Analyze anticipatory positioning (through zone demand patterns)
    zone_demands = {}
    for zone in system.zone_agents:
        zone_demands[zone.id] = zone.demand_forecast
    
    analysis['anticipatory_positioning'] = {
        'zones_with_positive_trend': sum(1 for z in zone_demands.values() if z.get('trend', 0) > 0),
        'average_demand_trend': np.mean([z.get('trend', 0) for z in zone_demands.values()]),
        'demand_variance': np.var([z.get('current', 0) for z in zone_demands.values()])
    }
    
    print(f"\n🎯 ANTICIPATORY POSITIONING:")
    print(f"  • Zones with increasing demand: {analysis['anticipatory_positioning']['zones_with_positive_trend']}")
    print(f"  • Average demand trend: {analysis['anticipatory_positioning']['average_demand_trend']:.2f}")
    print(f"  • Demand variance across zones: {analysis['anticipatory_positioning']['demand_variance']:.2f}")
    
    # Analyze adaptive coordination
    total_requests = simulation_results['requests_processed']
    successful_assignments = simulation_results['successful_assignments']
    avg_response_time = system.system_metrics['average_response_time']
    
    analysis['adaptive_coordination'] = {
        'success_rate': successful_assignments / total_requests,
        'avg_response_time_ms': avg_response_time,
        'bids_per_request': np.mean([r['bids_received'] for r in simulation_results['processing_results']]),
        'negotiation_efficiency': successful_assignments / max(total_requests, 1) / max(avg_response_time/1000, 1)
    }
    
    print(f"\n🔄 ADAPTIVE COORDINATION:")
    print(f"  • Success rate: {analysis['adaptive_coordination']['success_rate']:.1%}")
    print(f"  • Average response time: {analysis['adaptive_coordination']['avg_response_time_ms']:.1f}ms")
    print(f"  • Average bids per request: {analysis['adaptive_coordination']['bids_per_request']:.1f}")
    print(f"  • Negotiation efficiency: {analysis['adaptive_coordination']['negotiation_efficiency']:.3f}")
    
    # Analyze system resilience
    active_vehicles = sum(1 for v in system.vehicle_agents if len(v.assigned_requests) > 0)
    total_capacity = sum(v.capacity for v in system.vehicle_agents)
    used_capacity = sum(v.capacity - v.remaining_capacity for v in system.vehicle_agents)
    
    analysis['system_resilience'] = {
        'active_vehicle_ratio': active_vehicles / len(system.vehicle_agents),
        'capacity_utilization': used_capacity / total_capacity,
        'system_throughput': successful_assignments,
        'scalability_factor': len(system.vehicle_agents) * len(system.zone_agents)
    }
    
    print(f"\n🛡️  SYSTEM RESILIENCE:")
    print(f"  • Active vehicle ratio: {analysis['system_resilience']['active_vehicle_ratio']:.1%}")
    print(f"  • Capacity utilization: {analysis['system_resilience']['capacity_utilization']:.1%}")
    print(f"  • System throughput: {analysis['system_resilience']['system_throughput']} requests")
    print(f"  • Scalability factor: {analysis['system_resilience']['scalability_factor']}")
    
    return analysis

# Analyze emergent optimization
emergent_analysis = analyze_emergent_optimization(system)

In [ ]:
def distributed_intelligence_summary():
    """Provide comprehensive summary of distributed intelligence approach"""
    
    print("\n" + "="*80)
    print("DYNAMIC VEHICLE ROUTING PROBLEM - DISTRIBUTED INTELLIGENCE SUMMARY")
    print("="*80)
    
    print("\n🤖 MULTI-AGENT ARCHITECTURE:")
    print("  • Vehicle Agents: Local routing intelligence and real-time awareness")
    print("  • Zone Agents: Geographical management and demand forecasting")
    print("  • Request Agents: Autonomous service demand representation")
    print("  • Infrastructure Agents: Real-time condition monitoring")
    
    print("\n🤝 CONTRACT NET PROTOCOL:")
    print("  • Decentralized request assignment through competitive bidding")
    print("  • Multi-criteria bid evaluation with confidence scoring")
    print("  • Automated negotiation and conflict resolution")
    print("  • Real-time coordination without central bottleneck")
    
    print("\n📊 SYSTEM PERFORMANCE:")
    print(f"  • Total requests processed: {simulation_results['requests_processed']}")
    print(f"  • Successful assignments: {simulation_results['successful_assignments']}")
    print(f"  • Success rate: {simulation_results['successful_assignments']/simulation_results['requests_processed']:.1%}")
    print(f"  • Average response time: {system.system_metrics['average_response_time']:.1f}ms")
    print(f"  • Vehicle utilization: {system.system_metrics['vehicle_utilization']:.1%}")
    
    print("\n🌟 EMERGENT OPTIMIZATION PATTERNS:")
    print("  • Dynamic Load Balancing: Automatic workload distribution")
    print("  • Anticipatory Positioning: Demand-based vehicle guidance")
    print("  • Adaptive Coordination: Real-time negotiation optimization")
    print("  • System Resilience: No single point of failure")
    
    print("\n🚀 SCALABILITY ADVANTAGES:")
    print("  • Linear scalability with number of zones and vehicles")
    print("  • Constant-time local decisions regardless of system size")
    print("  • Natural load balancing through distributed bidding")
    print("  • Graceful degradation under individual agent failures")
    
    print("\n⚡ REAL-TIME CAPABILITIES:")
    print(f"  • Negotiation overhead: {system.system_metrics['average_response_time']:.1f}ms average")
    print("  • Immediate response to new requests")
    print("  • Continuous system adaptation and optimization")
    print("  • No central coordination bottleneck")
    
    print("\n🔄 COMPARISON WITH PREVIOUS TIERS:")
    print("  • vs Tier 1 (Mathematical): Handles 15+ vehicles vs. ≤5 vehicles")
    print("  • vs Tier 4 (Neuroevolution): Distributed vs. centralized learning")
    print("  • Response time: Milliseconds vs. seconds/minutes")
    print("  • Scalability: Linear vs. exponential complexity")
    print("  • Resilience: High vs. single point of failure")
    
    print("\n🎯 PRACTICAL APPLICATIONS:")
    print("  • Large-scale metropolitan delivery networks")
    print("  • Ride-sharing and transportation platforms")
    print("  • Emergency response and disaster relief")
    print("  • Supply chain and logistics coordination")
    
    print("\n🎓 EDUCATIONAL VALUE:")
    print("  • Demonstrates emergent intelligence from local interactions")
    print("  • Shows scalability through distributed coordination")
    print("  • Illustrates multi-agent negotiation protocols")
    print("  • Provides foundation for autonomous system design")
    
    print("\n" + "="*80)

# Display comprehensive summary
distributed_intelligence_summary()